# Take note to install requirements.txt first
### NOTE: Python needs to be 3.12, basically more than Python3.11
### This code is for the new gradesheet not LM TIMS

## Install libraries

In [15]:
import joblib
from pathlib import Path
import pandas as pd
import numpy as np

## Read raw gradesheet data, pivot data

In [ ]:
df = pd.read_csv("../data/raw_gradesheets/GradesheetConsolidated.csv") #take note on file directive, from data> raw_gradesheets

selected_columns = ["_cr0a6_gradesheettype_value@OData.Community.Display.V1.FormattedValue", "cr0a6_finalscore", "_cr0a6_trainee_value@OData.Community.Display.V1.FormattedValue",'cr0a6_completionstatus']
df = df[selected_columns]


df.columns = ["Module", "Score", "Student", "Status"]

# Completion flag
df["CompletedFlag"] = (df["Status"] == "Completed").astype(int)

# Score (only completed)
df_completed = df[df["CompletedFlag"] == 1]

score_pivot = df_completed.pivot_table(
    index="Student",
    columns="Module",
    values="Score",
    aggfunc="max"
)

completion_pivot = df.pivot_table(
    index="Student",
    columns="Module",
    values="CompletedFlag",
    aggfunc="max"
)

# Combine
df_final = score_pivot.reset_index()

df_final = df_final.rename(columns={"Student": "STUDENT_ID"})
df_final.columns.name = None

print(df_final.head(20))

       STUDENT_ID  GH 1  GH 1 (W)  GH 10  GH 10 (W)  GH 11  GH 11^+ (W)  \
0     208.BHAVESH   NaN       NaN   5.41        NaN   5.29          NaN   
1        208.HOJ+  5.81       NaN    NaN        NaN    NaN          NaN   
2       208.JKOK+  5.96       NaN    NaN        NaN    NaN          NaN   
3        208.KOHS  5.75       NaN    NaN        NaN    NaN          NaN   
4        208.KOHZ  4.58       NaN    NaN        NaN    NaN          NaN   
5       208.LAIL+  4.86       NaN    NaN        NaN    NaN          NaN   
6      208.NGOOIJ  5.26       NaN    NaN        NaN    NaN          NaN   
7       208.RLEE+  5.52       NaN   4.85        NaN    NaN          NaN   
8        208.TANH  5.76       NaN    NaN        NaN    NaN          NaN   
9       208.TANJ+  5.13       NaN    NaN        NaN    NaN          NaN   
10       208.TANY  5.46       NaN    NaN        NaN    NaN          NaN   
11       208.TEOC  5.61       NaN    NaN        NaN    NaN          NaN   
12       208.TOHZ  5.23  

## Removal of WSO student, Batch 208= 16 trainees

In [3]:
df_clean = df_final[~df_final["STUDENT_ID"].str.contains(r"^\d+w", regex=True)]

print(df_clean.head(30))

     STUDENT_ID  GH 1  GH 1 (W)  GH 10  GH 10 (W)  GH 11  GH 11^+ (W)  GH 12  \
0   208.BHAVESH   NaN       NaN   5.41        NaN   5.29          NaN   4.97   
1      208.HOJ+  5.81       NaN    NaN        NaN    NaN          NaN    NaN   
2     208.JKOK+  5.96       NaN    NaN        NaN    NaN          NaN    NaN   
3      208.KOHS  5.75       NaN    NaN        NaN    NaN          NaN    NaN   
4      208.KOHZ  4.58       NaN    NaN        NaN    NaN          NaN    NaN   
5     208.LAIL+  4.86       NaN    NaN        NaN    NaN          NaN    NaN   
6    208.NGOOIJ  5.26       NaN    NaN        NaN    NaN          NaN    NaN   
7     208.RLEE+  5.52       NaN   4.85        NaN    NaN          NaN    NaN   
8      208.TANH  5.76       NaN    NaN        NaN    NaN          NaN    NaN   
9     208.TANJ+  5.13       NaN    NaN        NaN    NaN          NaN    NaN   
10     208.TANY  5.46       NaN    NaN        NaN    NaN          NaN    NaN   
11     208.TEOC  5.61       NaN    NaN  

## Dropping columns for WSO-related & SGH Modules


In [4]:
cols_to_drop = df_clean.columns[df_clean.columns.str.contains(r"\(W\)")]
print("Columns to drop:")
print(cols_to_drop)

Columns to drop:
Index(['GH 1 (W)', 'GH 10 (W)', 'GH 11^+ (W)', 'GH 12^+ (W)', 'GH 2 (W)',
       'GH 3 (W)', 'GH 4 (W)', 'GH 5 (W)', 'GH 6 (W)', 'GH 7 (W)', 'GH 8* (W)',
       'GH 9+ (W)', 'OFS 8 (W)', 'W-GHT^+ (W)'],
      dtype='str')


In [5]:
# Drop columns that contain '(W)'
df_clean = df_clean.loc[:, ~df_clean.columns.str.contains(r"\(W\)")]

# Drop columns SGH 1, SGH 2, SGH 3

df_clean = df_clean.drop(columns=["SGH 1", "SGH 2", "SGH 3"])

print(df_clean.head(30))


     STUDENT_ID  GH 1  GH 10  GH 11  GH 12  GH 14  GH 15  GH 19  GH 2  GH 21  \
0   208.BHAVESH   NaN   5.41   5.29   4.97   5.35   4.82   4.44  6.26   4.55   
1      208.HOJ+  5.81    NaN    NaN    NaN    NaN    NaN    NaN  5.33    NaN   
2     208.JKOK+  5.96    NaN    NaN    NaN    NaN    NaN    NaN  6.31    NaN   
3      208.KOHS  5.75    NaN    NaN    NaN    NaN    NaN    NaN  5.21    NaN   
4      208.KOHZ  4.58    NaN    NaN    NaN    NaN    NaN    NaN  5.50    NaN   
5     208.LAIL+  4.86    NaN    NaN    NaN    NaN    NaN    NaN  4.49    NaN   
6    208.NGOOIJ  5.26    NaN    NaN    NaN    NaN    NaN    NaN  5.47    NaN   
7     208.RLEE+  5.52   4.85    NaN    NaN    NaN    NaN    NaN  5.66    NaN   
8      208.TANH  5.76    NaN    NaN    NaN    NaN    NaN    NaN  5.94    NaN   
9     208.TANJ+  5.13    NaN    NaN    NaN    NaN    NaN    NaN  5.13    NaN   
10     208.TANY  5.46    NaN    NaN    NaN    NaN    NaN    NaN  4.62    NaN   
11     208.TEOC  5.61    NaN    NaN    N

# Reindex Columns 

### Missing Modules:

Module_1:
GH 13 , GH 17

Module_2:
GH 24, GH 28, GH 30, GH 31, GH 32, GH 33, GH 34, GHT

Module_3:
IF 1, IF 2, IF 3, IF 4, IF 5, IF 6, IFT


## **POC for Batch 208, hotfix, filling up missing columns with NaN value

In [ ]:
# Define modules
MODULE_1 = [
    "GH 1","GH 2","GH 3","GH 4","GH 5","GH 6","GH 7","GH 8","GH 9",
    "GH 10","GH 11","GH 12","GH 13","GH 14","GH 15","GH 17","GH 19","GH 21","GH 22"
]

MODULE_2 = [
    "GH 24","GH 25","GH 27","GH 28","GH 29","GH 30","GH 31","GH 32","GH 33","GH 34","GHT"
]

MODULE_3 = ["IF 1","IF 2","IF 3","IF 4","IF 5","IF 6","IFT"]

LESSONS_TO_KEEP = MODULE_1 + MODULE_2 + MODULE_3

# Final column order
final_columns = ["STUDENT_ID"] + LESSONS_TO_KEEP

df_clean = df_clean.reindex(columns=final_columns)

df_clean.head(20)

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GH 33,GH 34,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT
0,208.BHAVESH,NaN,6.26,NaN,NaN,5.33,NaN,4.92,4.73,4.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,208.HOJ+,5.81,5.33,5.36,4.63,4.94,4.62,4.80,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,208.JKOK+,5.96,6.31,4.19,4.65,4.41,4.45,4.26,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,208.KOHS,5.75,5.21,4.40,4.65,4.91,4.73,4.43,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,208.KOHZ,4.58,5.50,4.12,4.83,4.49,4.53,4.76,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,208.LAIL+,4.86,4.49,4.04,4.53,4.54,4.18,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,208.NGOOIJ,5.26,5.47,4.44,5.07,4.41,4.65,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,208.RLEE+,5.52,5.66,5.26,5.47,5.08,4.94,4.70,4.92,5.01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,208.TANH,5.76,5.94,4.98,5.11,4.57,4.67,4.77,4.47,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,208.TANJ+,5.13,5.13,4.17,4.94,4.95,4.55,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Run Inference

## Load trained model

In [14]:
# Load trained model
stage1_best_model = joblib.load("../models/stage1_best_model.joblib")
stage2_best_model = joblib.load("../models/stage2_best_model.joblib")

### Stage 1: Predict BWC pass or fail

In [16]:
# Keep only the feature columns
X1 = df_clean.drop(columns="STUDENT_ID")

# Get predictions and associated probability
y_pred1 = stage1_best_model.predict(X1)
y_score1 = stage1_best_model.predict_proba(X1)[:, 1]

# Append back to wide span table
df_clean["pred_bwc"] = y_pred1
df_clean["prob_bwc"] = y_score1
assert df_clean["pred_bwc"].isnull().sum() == 0
df_clean

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,pred_bwc,prob_bwc
0,208.BHAVESH,NaN,6.26,NaN,NaN,5.33,NaN,4.92,4.73,4.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.524005
1,208.HOJ+,5.81,5.33,5.36,4.63,4.94,4.62,4.80,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.140119
2,208.JKOK+,5.96,6.31,4.19,4.65,4.41,4.45,4.26,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.149697
3,208.KOHS,5.75,5.21,4.40,4.65,4.91,4.73,4.43,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.026282
4,208.KOHZ,4.58,5.50,4.12,4.83,4.49,4.53,4.76,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.018055
5,208.LAIL+,4.86,4.49,4.04,4.53,4.54,4.18,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.014416
6,208.NGOOIJ,5.26,5.47,4.44,5.07,4.41,4.65,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.018316
7,208.RLEE+,5.52,5.66,5.26,5.47,5.08,4.94,4.70,4.92,5.01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.217273
8,208.TANH,5.76,5.94,4.98,5.11,4.57,4.67,4.77,4.47,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.154953
9,208.TANJ+,5.13,5.13,4.17,4.94,4.95,4.55,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.020260


### Stage 2: Among predicted BWC passes, predict fighter or not

In [17]:
# Keep only predicted passes
passes = df_clean[df_clean["pred_bwc"] == 1].drop(columns=["pred_bwc", "prob_bwc"])

# Keep only the feature columns
X2 = passes.drop(columns=["STUDENT_ID"])

# Get predictions and associated probability
y_pred2 = stage2_best_model.predict(X2)
y_score2 = stage2_best_model.predict_proba(X2)[:, 1]

# Append back to wide span table
passes["pred_fighter"] = y_pred2
passes["prob_fighter"] = y_score2
assert passes["pred_fighter"].isnull().sum() == 0
passes

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,pred_fighter,prob_fighter
0,208.BHAVESH,NaN,6.26,NaN,NaN,5.33,NaN,4.92,4.73,4.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.912636


### Final output

In [18]:
predictions = pd.merge(
    df_clean[["STUDENT_ID", "pred_bwc", "prob_bwc"]],
    passes[["STUDENT_ID", "pred_fighter", "prob_fighter"]],
    on="STUDENT_ID",
    how="left",
)
predictions = predictions.fillna(0.0)

In [19]:
predictions.head(20)

,STUDENT_ID,pred_bwc,prob_bwc,pred_fighter,prob_fighter
0,208.BHAVESH,1.0,0.524005,1.0,0.912636
1,208.HOJ+,0.0,0.140119,0.0,0.000000
2,208.JKOK+,0.0,0.149697,0.0,0.000000
3,208.KOHS,0.0,0.026282,0.0,0.000000
4,208.KOHZ,0.0,0.018055,0.0,0.000000
5,208.LAIL+,0.0,0.014416,0.0,0.000000
6,208.NGOOIJ,0.0,0.018316,0.0,0.000000
7,208.RLEE+,0.0,0.217273,0.0,0.000000
8,208.TANH,0.0,0.154953,0.0,0.000000
9,208.TANJ+,0.0,0.020260,0.0,0.000000


## Saving into predictions folder

In [20]:
predictions.to_csv("../predictions/208_predictions.csv", index=False)